# Notebook 4 — Inference Demo

Run on **Google Colab** (free T4 GPU) to test the fine-tuned IAM Policy Generator.

**Before running:** upload `iam-policy-adapter/` to Google Drive at `MyDrive/iam-policy-llm/iam-policy-adapter/`

In [1]:
# Run once
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

## 1. Load adapter from Google Drive

In [2]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')

src = '/content/drive/MyDrive/iam-policy-llm/iam-policy-adapter'
dst = '/content/iam-policy-adapter'

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f'Adapter copied to {dst}')
print(os.listdir(dst))

Mounted at /content/drive
Adapter copied to /content/iam-policy-adapter
['adapter_config.json', 'chat_template.jinja', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors', 'tokenizer.json']


## 2. Load model

In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = '/content/iam-policy-adapter',
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print('Model ready')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.5 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model ready


## 3. Generate a policy

In [4]:
import json

def generate_policy(requirement: str) -> dict:
    prompt = (
        "### Instruction:\nGenerate an AWS IAM policy for the following requirement\n\n"
        f"### Input:\n{requirement}\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.1,
        do_sample=False,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response_text = decoded.split('### Response:')[-1].strip()
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        import re
        m = re.search(r'\{.*\}', response_text, re.DOTALL)
        return json.loads(m.group()) if m else {'raw_output': response_text}


requirement = (
    "The finance team needs read-only access to the S3 bucket 'prod-invoices'. "
    "No write or delete permissions. MFA must be required."
)

result = generate_policy(requirement)
print(json.dumps(result, indent=2))

Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

{
  "policy": {
    "Version": "2012-10-17",
    "Statement": [
      {
        "Effect": "Allow",
        "Action": [
          "s3:GetObject",
          "s3:ListBucket",
          "s3:GetBucketLocation"
        ],
        "Resource": [
          "arn:aws:s3:::prod-invoices",
          "arn:aws:s3:::prod-invoices/*"
        ],
        "Condition": {
          "Bool": {
            "aws:MultiFactorAuthPresent": "true"
          }
        }
      }
    ]
  },
  "nist_controls": [
    "AC-3",
    "AC-6",
    "SC-7"
  ],
  "risk_notes": "MFA requirement prevents unauthorized data access. Resource scope limited to the specific bucket and its contents."
}


## 4. Try more examples

In [5]:
examples = [
    "DevOps team can start and stop EC2 instances in eu-west-1 only. No terminate or security group changes.",
    "The data pipeline Lambda function needs to read from DynamoDB table 'orders' and write to S3 bucket 'pipeline-output'. No delete permissions.",
    "Security auditors need read-only access to CloudTrail logs and Config rules across all regions. MFA required.",
    "The CI/CD pipeline role needs to push images to ECR, deploy to ECS, and update CloudFormation stacks. Deny iam:PassRole to any resource.",
]

for req in examples:
    print(f'\n--- Requirement ---\n{req}')
    result = generate_policy(req)
    policy = result.get('policy', {})
    nist   = result.get('nist_controls', [])
    notes  = result.get('risk_notes', '')
    stmts  = policy.get('Statement', [])
    effects = [s.get('Effect') for s in stmts if isinstance(s, dict)]
    print(f'Statements: {len(stmts)} ({effects})')
    print(f'NIST: {nist}')
    print(f'Risk: {notes}')

Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Requirement ---
DevOps team can start and stop EC2 instances in eu-west-1 only. No terminate or security group changes.


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Statements: 1 (['Allow'])
NIST: ['CM-3', 'AC-3', 'SC-28']
Risk: Region restriction prevents cross-region instance management. Start and stop actions are explicitly allowed, while terminate and security group modifications are explicitly denied.

--- Requirement ---
The data pipeline Lambda function needs to read from DynamoDB table 'orders' and write to S3 bucket 'pipeline-output'. No delete permissions.


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Statements: 2 (['Allow', 'Allow'])
NIST: ['AC-3', 'AC-6', 'SC-7']
Risk: Region restriction ensures data is processed within the designated geographic boundary.

--- Requirement ---
Security auditors need read-only access to CloudTrail logs and Config rules across all regions. MFA required.


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Statements: 2 (['Allow', 'Allow'])
NIST: ['AU-6', 'AU-12', 'AC-6']
Risk: MFA requirement ensures that only authenticated users can access audit data.

--- Requirement ---
The CI/CD pipeline role needs to push images to ECR, deploy to ECS, and update CloudFormation stacks. Deny iam:PassRole to any resource.
Statements: 3 (['Allow', 'Allow', 'Allow'])
NIST: ['AC-3', 'AC-6', 'CM-6']
Risk: Attribute-based access control (ABAC) ensures that only personnel with the 'DevOps' tag can perform pipeline operations.


## 5. (Optional) Quick Gradio UI

In [6]:
!pip install gradio -q

import gradio as gr

def ui_generate(requirement):
    result = generate_policy(requirement)
    return (
        json.dumps(result.get('policy', {}), indent=2),
        ', '.join(result.get('nist_controls', [])),
        result.get('risk_notes', ''),
    )

demo = gr.Interface(
    fn=ui_generate,
    inputs=gr.Textbox(lines=4, label='Access Control Requirement'),
    outputs=[
        gr.Code(language='json', label='IAM Policy'),
        gr.Textbox(label='NIST SP 800-53 Controls'),
        gr.Textbox(label='Risk Notes'),
    ],
    title='IAM Policy Generator (Fine-tuned Llama 3.2 3B)',
    description='Generates AWS IAM policies from plain-English requirements. **Review all outputs before production use.**',
    examples=[
        ['Finance team needs read-only access to S3 bucket prod-invoices. No delete. MFA required.'],
        ['DevOps can start and stop EC2 instances in eu-west-1 only. No terminate or security group changes.'],
        ['Security auditors need read-only CloudTrail and Config access. MFA required.'],
    ],
    flagging_mode='never',
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9a52a05a27ef7eabb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
